# 🤖 AutoML Systémique avec Optimisation Multi-Objectif

## Objectif
Ce notebook vous guide étape par étape dans la compréhension et l'utilisation du framework AutoML.

**Ce qu'on optimise simultanément :**
- 🎯 **Accuracy** : performance prédictive
- ⚡ **Latence** : temps d'inférence
- 💾 **Mémoire** : RAM utilisée

**Comment :**
- **SMAC3** : optimisation bayésienne (apprend des évaluations passées)
- **Hyperband** : allocation budgétaire adaptative (arrête tôt les mauvaises configs)
- **Méta-apprentissage** : warm-start depuis les datasets précédents

## 📦 Installation des dépendances

In [ ]:
# Installer toutes les dépendances
!pip install smac scikit-learn openml xgboost plotly streamlit ConfigSpace scipy tqdm -q
print('✅ Installation terminée')

## 🏗️ Structure du projet

In [ ]:
import os
# Afficher la structure
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git', 'smac_output', 'results']]
    level = root.replace('.', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        if file.endswith('.py') or file.endswith('.txt'):
            print(f'{subindent}📄 {file}')

## Module 1 : Chargement des datasets

OpenML est une plateforme avec des milliers de datasets standardisés, parfait pour comparer des algorithmes.

In [ ]:
import sys
sys.path.insert(0, '.')

from data.datasets import load_dataset, get_dataset_summary, DATASETS

print('Datasets disponibles :')
for name, (id_, readable, task) in DATASETS.items():
    print(f'  - {name:20s} → {readable} (OpenML id={id_})')

In [ ]:
# Charger un dataset
data = load_dataset('diabetes')
print(f'Dataset : {data["name"]}')
print(f'X_train : {data["X_train"].shape}')
print(f'y_train : {data["y_train"].shape}')
print(f'Classes : {data["n_classes"]}')

## Module 2 : Espace de recherche

SMAC explore cet espace pour trouver les meilleures combinaisons (algorithme + hyperparamètres).

In [ ]:
from models.search_space import get_configspace, build_model

cs = get_configspace()
print(f'Nombre de hyperparamètres : {len(cs.get_hyperparameters())}')
print('\nQuelques hyperparamètres :')
for hp in list(cs.get_hyperparameters())[:8]:
    print(f'  {hp.name}')

In [ ]:
# Générer quelques configurations aléatoires
print('Exemples de configurations aléatoires :')
for i in range(3):
    cfg = cs.sample_configuration()
    model = build_model(dict(cfg))
    print(f'\nConfig {i+1}: {dict(cfg)["algorithm"]}')
    print(f'  Modèle: {model}')

## Module 3 : Évaluation multi-objectif

Pour chaque configuration, on mesure les 3 objectifs simultanément.

In [ ]:
from evaluation.evaluator import evaluate_configuration
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=50, random_state=42)

result = evaluate_configuration(
    model   = model,
    X_train = data['X_train'],
    y_train = data['y_train'],
    X_test  = data['X_test'],
    y_test  = data['y_test'],
)

print('📊 Résultats de l\'évaluation :')
print(f'  Accuracy   : {result["accuracy"]:.4f}')
print(f'  Latence    : {result["latency_ms"]:.2f} ms')
print(f'  Mémoire    : {result["memory_mb"]:.2f} MB')
print(f'  Statut     : {result["status"]}')

## Module 4 : SMAC + Hyperband

### Pourquoi SMAC ?
- **GridSearch** : teste **toutes** les combinaisons → 500 n_estimators × 50 max_features = 25000 évaluations !
- **SMAC** : apprend quelles zones de l'espace sont prometteuses → 50 évaluations suffisent

### Pourquoi Hyperband ?
- Certaines configs sont clairement mauvaises même sur 10% des données
- Hyperband les élimine tôt → on économise beaucoup de calcul

In [ ]:
from optimization.smac_optimizer import AutoMLOptimizer

optimizer = AutoMLOptimizer(
    dataset_name = 'diabetes',
    n_trials     = 20,       # 20 évaluations (peu pour le test)
    min_budget   = 0.1,      # commence avec 10% des données
    max_budget   = 1.0,      # finit avec 100%
    output_dir   = '/tmp/smac_notebook',
)

best = optimizer.run(
    X_train = data['X_train'],
    y_train = data['y_train'],
    X_test  = data['X_test'],
    y_test  = data['y_test'],
)

print(f'\n🏆 Meilleure config trouvée :')
print(f'  Algorithme : {optimizer.best_config.get("algorithm")}')
print(f'  Accuracy   : {best["accuracy"]:.4f}')
print(f'  Latence    : {best["latency_ms"]:.2f} ms')
print(f'  Mémoire    : {best["memory_mb"]:.2f} MB')

In [ ]:
# Visualiser les résultats
df_results = optimizer.get_all_results()
print(f'Évaluations totales : {len(df_results)}')
df_results[['algorithm', 'accuracy', 'latency_ms', 'memory_mb', 'score', 'budget']].head(10)

## Module 5 : Méta-apprentissage

In [ ]:
from optimization.meta_learning import MetaLearningBase, extract_meta_features

meta_base = MetaLearningBase(storage_path='/tmp/meta_test.pkl')

# Extraire les méta-features
meta_feats = extract_meta_features(data['X_train'], data['y_train'])
print('Méta-features extraites :')
for k, v in list(meta_feats.items())[:6]:
    print(f'  {k}: {v:.4f}')

# Ajouter dans la base
meta_base.add_entry(
    dataset_name  = 'diabetes',
    meta_features = meta_feats,
    best_config   = optimizer.best_config or {},
    best_score    = optimizer.best_score,
    best_accuracy = best['accuracy'],
)

print('\n📚 Base méta :')
print(meta_base.get_summary())

## Module 6 : Visualisations

In [ ]:
from analysis.visualizations import (
    plot_pareto_front_3d,
    plot_convergence,
    plot_algorithm_comparison,
    plot_tradeoff_scatter
)

df_results = optimizer.get_all_results()

# Front de Pareto
fig1 = plot_pareto_front_3d(df_results, 'Front de Pareto - Diabetes')
fig1.show()

# Convergence
fig2 = plot_convergence(df_results, 'Convergence SMAC')
fig2.show()

# Compromis
fig3 = plot_tradeoff_scatter(df_results)
fig3.show()

## 🔄 Pipeline complet sur plusieurs datasets

Cette cellule lance le pipeline sur 3 datasets pour démonstration.

In [ ]:
from run_pipeline import run_full_pipeline

# Lancer sur 3 datasets avec peu de trials (rapide)
results = run_full_pipeline(
    dataset_names = ['diabetes', 'breast_cancer', 'iris'],
    config = {
        'n_trials': 10,
        'min_budget': 0.1,
        'max_budget': 1.0,
        'output_dir': '/tmp/smac_full',
        'results_dir': '/tmp/results',
        'meta_base_path': '/tmp/meta_full.pkl',
        'seed': 42,
    }
)

print('\n✅ Pipeline terminé !')
results

## 🚀 Lancer l'application Streamlit

Pour lancer l'interface graphique complète :

In [ ]:
# Dans votre terminal, depuis le dossier automl_project :
print('Commande pour lancer Streamlit :')
print('streamlit run app/streamlit_app.py')
print()
print('Ou depuis le dossier parent :')
print('streamlit run automl_project/app/streamlit_app.py')